In [ ]:
from pulp import *

# 선형계획법(Linear Programming) 문제 정의: 제과점 이익 최대화
LP = LpProblem("bakery_problem", LpMaximize)

# 변수 선언: X1(케이크 수), X2(바게트 수)
X1 = LpVariable("cake")
X2 = LpVariable("baguette")

# 목적함수: 10*X1 + 6*X2 (총 이익 최대화)
LP += 10*X1 + 6*X2

# 제약조건 1: 3*X1 + 8*X2 <= 20 (오븐 사용 시간)
LP += 3*X1 + 8*X2 <= 20

# 제약조건 2: 45*X1 + 30*X2 <= 180 (밀가루 사용량)
LP += 45*X1 + 30*X2 <= 180

# 변수의 비음수 조건(생산량은 0 이상)
LP += X1 >= 0
LP += X2 >= 0

# 최적해 계산
LP.solve()

# 결과 출력: 각 변수의 최적값과 총 이익
for v in LP.variables():
    print(v.name, '=', v.varValue)
print("Total profit is ", value(LP.objective))

In [ ]:
import numpy as np

# 시뮬레이션 횟수
n = 100000

# 온도 t는 0°C에서 30°C까지 uniform 분포로 변동
temp = np.random.uniform(0, 30, n)

# 날씨는 4/5의 확률로 맑고(w=0), 1/5의 확률로 흐림(w=1)
weather = np.random.binomial(1, 0.2, n)  # 0.2확률로 흐림(w=1), 0.8확률로 맑음(w=0)

# 수요 계산
demand = np.zeros(n, dtype='float32')
for i in range(n):
    # 새로운 평균 수요 공식 적용
    if weather[i] == 0:  # 맑은 날 (w=0)
        mu_d = 100 + 3 * temp[i]
        sigma_d = 0.5 * mu_d
    else:  # 흐린 날 (w=1)
        mu_d = 30 + 4 * temp[i]
        sigma_d = 0.6 * mu_d

    d = np.random.normal(mu_d, sigma_d)
    if d < 0:
        d = 0
    demand[i] = d

# 비용 및 가격 설정 (기존 예제와 동일)
p = 8000  # 판매가격
c = 2000  # 생산비용
s = 0     # 폐기비용
Cu = p - c  # 기회비용 (품절비용)
Co = c - s  # 보유비용 (과잉재고비용)

print("=== 기본 정보 ===")
print(f"시뮬레이션 횟수: {n:,}")
print(f"온도 범위: 0°C ~ 30°C (uniform 분포)")
print(f"날씨 확률: 맑음 80%, 흐림 20%")
print(f"판매가격: {p:,}원")
print(f"생산비용: {c:,}원")
print(f"품절비용: {Cu:,}원")
print(f"재고비용: {Co:,}원")

# 평균 수요 분석
sunny_demand = []
cloudy_demand = []
for i in range(n):
    if weather[i] == 0:
        sunny_demand.append(demand[i])
    else:
        cloudy_demand.append(demand[i])

print(f"\n=== 수요 분석 ===")
print(f"전체 평균 수요: {np.mean(demand):.2f}")
print(f"맑은 날 평균 수요: {np.mean(sunny_demand):.2f} (샘플 수: {len(sunny_demand):,})")
print(f"흐린 날 평균 수요: {np.mean(cloudy_demand):.2f} (샘플 수: {len(cloudy_demand):,})")

# 단순 평균 기반 생산량 결정
sum_profit_simple = 0
sum_Q_simple = 0

print("\n=== 단순 평균 기반 생산량 결정 ===")
for i in range(n):
    # 단순히 평균 수요로 생산량 결정
    if weather[i] == 0:  # 맑은 날
        Q = 100 + 3 * temp[i]
    else:  # 흐린 날
        Q = 30 + 4 * temp[i]

    d = demand[i]

    # 이익 계산
    profit = -c * Q
    if d <= Q:
        profit += p * d + s * (Q - d)
    else:
        profit += p * Q

    sum_Q_simple += Q
    sum_profit_simple += profit

avg_Q_simple = sum_Q_simple / n
avg_profit_simple = sum_profit_simple / n

print(f"평균 생산량: {avg_Q_simple:.2f}")
print(f"평균 기대이익: {avg_profit_simple:.2f}원")

# 최적 생산량 결정 (뉴스벤더 모델 적용)
print("\n=== 최적 생산량 결정 (뉴스벤더 모델) ===")

# 임계확률 계산
critical_ratio = Cu / (Cu + Co)
print(f"임계확률: {critical_ratio:.3f}")

sum_profit_optimal = 0
sum_Q_optimal = 0

for i in range(n):
    # 날씨와 온도에 따른 수요 분포 계산
    if weather[i] == 0:  # 맑은 날
        mu_d = 100 + 3 * temp[i]
        sigma_d = 0.5 * mu_d
    else:  # 흐린 날
        mu_d = 30 + 4 * temp[i]
        sigma_d = 0.6 * mu_d

    # 정규분포의 역함수를 이용한 최적 생산량 계산
    from scipy.stats import norm
    Q_optimal = norm.ppf(critical_ratio, mu_d, sigma_d)

    if Q_optimal < 0:
        Q_optimal = 0

    d = demand[i]

    # 이익 계산
    profit = -c * Q_optimal
    if d <= Q_optimal:
        profit += p * d + s * (Q_optimal - d)
    else:
        profit += p * Q_optimal

    sum_Q_optimal += Q_optimal
    sum_profit_optimal += profit

avg_Q_optimal = sum_Q_optimal / n
avg_profit_optimal = sum_profit_optimal / n

print(f"평균 생산량: {avg_Q_optimal:.2f}")
print(f"평균 기대이익: {avg_profit_optimal:.2f}원")

# 비교 분석
print(f"\n=== 비교 분석 ===")
print(f"단순 평균 방법 - 생산량: {avg_Q_simple:.2f}, 이익: {avg_profit_simple:.2f}원")
print(f"최적화 방법   - 생산량: {avg_Q_optimal:.2f}, 이익: {avg_profit_optimal:.2f}원")
print(f"이익 개선: {avg_profit_optimal - avg_profit_simple:.2f}원 ({((avg_profit_optimal - avg_profit_simple) / avg_profit_simple * 100):.2f}%)")

# 온도별 수요 분석
temp_ranges = [(0, 10), (10, 20), (20, 30)]
print(f"\n=== 온도별 수요 분석 ===")

for temp_min, temp_max in temp_ranges:
    temp_mask = (temp >= temp_min) & (temp < temp_max)
    temp_demand = demand[temp_mask]
    temp_weather = weather[temp_mask]

    sunny_temp_demand = temp_demand[temp_weather == 0]
    cloudy_temp_demand = temp_demand[temp_weather == 1]

    print(f"온도 {temp_min}°C-{temp_max}°C:")
    print(f"  전체 평균 수요: {np.mean(temp_demand):.2f}")
    if len(sunny_temp_demand) > 0:
        print(f"  맑은 날 평균 수요: {np.mean(sunny_temp_demand):.2f}")
    if len(cloudy_temp_demand) > 0:
        print(f"  흐린 날 평균 수요: {np.mean(cloudy_temp_demand):.2f}")

In [1]:
# pip install pulp
from pulp import *

# 선형계획법(Linear Programming) 문제 정의: sausage_problem, 이익 최대화
LP = LpProblem("sausage_problem", LpMaximize)

# 변수 선언 (각 소시지 종류별 생산량, 0 이상)
Xcr = LpVariable("Xcr", 0)
Xbr = LpVariable("Xbr", 0)
Xpr = LpVariable("Xpr", 0)
Xar = LpVariable("Xar", 0)
Xcb = LpVariable("Xcb", 0)
Xbb = LpVariable("Xbb", 0)
Xpb = LpVariable("Xpb", 0)
Xab = LpVariable("Xab", 0)
Xcm = LpVariable("Xcm", 0)
Xbm = LpVariable("Xbm", 0)
Xpm = LpVariable("Xpm", 0)
Xam = LpVariable("Xam", 0)

# 목적함수: 총 이익 최대화
LP += (
    0.7*Xcr + 0.6*Xbr + 0.4*Xpr + 0.85*Xar
    + 1.05*Xcb + 0.95*Xbb + 0.75*Xpb + 1.20*Xab
    + 1.55*Xcm + 1.45*Xbm + 1.25*Xpm + 1.70*Xam
)

# 제약조건들
LP += Xcr + Xcb + Xcm <= 200
LP += Xbr + Xbb + Xbm <= 300
LP += Xpr + Xpb + Xpm <= 150
LP += Xar + Xab + Xam <= 400

LP += 0.9*Xbr + 0.9*Xpr - 0.1*Xcr - 0.1*Xar <= 0
LP += 0.8*Xcr - 0.2*Xbr - 0.2*Xpr - 0.2*Xar >= 0
LP += 0.25*Xbb - 0.75*Xcb - 0.75*Xpb - 0.75*Xab >= 0
LP += Xam == 0
LP += 0.5*Xbm + 0.5*Xpm - 0.5*Xcm - 0.5*Xam <= 0

# 최적해 계산
LP.solve()

# 결과 출력: 각 변수의 최적값과 총 이익
for v in LP.variables():
    print(v.name, '=', v.varValue)
print("Total profit is ", value(LP.objective))

ModuleNotFoundError: No module named 'pulp'

온도 t는 0𝑐 에서 30𝑐 까지 uniform 분포로 변동하며, 날씨는 4/5의 확률로 맑고(w=0), 1/5의 확률로
흐릴(w=1) 수 있도록 수정하시오.

In [5]:
import numpy as np
n=10000
temp = np.random.uniform(0, 30, n)
weather = np.random.binomial(1, 0.2, n)  # 0.2확률로 흐림(w=1), 0.8확률로 맑음(w=0)
demand = np.zeros(n, dtype='float32')
for i in range(n):
    # 새로운 평균 수요 공식 적용
    if weather[i] == 0:  # 맑은 날 (w=0)
        mu_d = 100 + 3 * temp[i]
        sigma_d = 0.5 * mu_d
    else:  # 흐린 날 (w=1)
        mu_d = 30 + 4 * temp[i]
        sigma_d = 0.6 * mu_d

    d = np.random.normal(mu_d, sigma_d)
    if d < 0:
        d = 0
    demand[i] = d

print(temp)
print(weather)
print(demand)





[ 1.8453412  15.18080806 19.57152292 ...  3.23513745 15.64403206
 13.87832832]
[1 0 0 ... 0 0 0]
[ 19.65772  77.78542 123.92349 ...   0.      158.66399 132.55153]


In [ ]:
import numpy as np

# TensorFlow 임포트 (에러 방지를 위해 try-except 사용)
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense
    TF_AVAILABLE = True
    print("TensorFlow를 사용하여 신경망 모델을 학습합니다.")
except ImportError:
    print("TensorFlow가 설치되지 않았습니다. 기본 통계 방법을 사용합니다.")
    TF_AVAILABLE = False

# 시뮬레이션 횟수
n = 100000

print("=== 냉면 수량 결정 시뮬레이션 ===")
print(f"시뮬레이션 횟수: {n:,}")

# 온도 t는 0°C에서 30°C까지 uniform 분포로 변동
temp = np.random.uniform(0, 30, n)

# 날씨는 4/5의 확률로 맑고(w=0), 1/5의 확률로 흐림(w=1)
weather = np.random.binomial(1, 0.2, n)  # 0.2확률로 흐림(w=1), 0.8확률로 맑음(w=0)

print(f"온도 범위: 0°C ~ 30°C (uniform 분포)")
print(f"날씨 확률: 맑음 80%, 흐림 20%")

# 수요 계산 (새로운 공식 적용)
demand = np.zeros(n, dtype='float32')
for i in range(n):
    # 새로운 평균 수요 공식 적용
    if weather[i] == 0:  # 맑은 날 (w=0)
        mu_d = 100 + 3 * temp[i]
        sigma_d = 0.5 * mu_d
    else:  # 흐린 날 (w=1)
        mu_d = 30 + 4 * temp[i]
        sigma_d = 0.6 * mu_d

    d = np.random.normal(mu_d, sigma_d)
    if d < 0:
        d = 0
    demand[i] = d

# 비용 및 가격 설정 (기존 예제와 동일)
p = 8000  # 판매가격
c = 2000  # 생산비용
s = 0     # 폐기비용
Cu = p - c  # 기회비용 (품절비용)
Co = c - s  # 보유비용 (과잉재고비용)

print(f"\n=== 비용 정보 ===")
print(f"판매가격: {p:,}원")
print(f"생산비용: {c:,}원")
print(f"품절비용: {Cu:,}원")
print(f"재고비용: {Co:,}원")

# 수요 분석
sunny_indices = weather == 0
cloudy_indices = weather == 1

print(f"\n=== 수요 분석 ===")
print(f"전체 평균 수요: {np.mean(demand):.2f}")
print(f"맑은 날 평균 수요: {np.mean(demand[sunny_indices]):.2f} (샘플 수: {np.sum(sunny_indices):,})")
print(f"흐린 날 평균 수요: {np.mean(demand[cloudy_indices]):.2f} (샘플 수: {np.sum(cloudy_indices):,})")

if TF_AVAILABLE:
    # 재고 손실 함수 정의
    def inventory_loss(demand_true, q):
        stock = q - demand_true
        is_understock = stock < 0
        inventory_error = tf.abs(stock)
        understock_loss = Cu * inventory_error
        overstock_loss = Co * inventory_error
        return tf.where(is_understock, understock_loss, overstock_loss)

    # 학습 데이터 준비
    x_train = np.zeros((n, 2), dtype='float32')
    x_train[:, 0] = (temp - min(temp)) / (max(temp) - min(temp))  # 온도 정규화
    x_train[:, 1] = weather  # 날씨 (0: 맑음, 1: 흐림)
    demand_train = (demand - min(demand)) / (max(demand) - min(demand))  # 수요 정규화

    # 신경망 모델 구성 (기존 예제와 동일한 구조)
    model = Sequential([
        Dense(2, input_shape=(2,), activation='sigmoid'),
        Dense(1, activation=None),
    ])

    # 모델 컴파일
    model.compile(optimizer='adam', loss=inventory_loss)

    # 모델 학습 (기존 예제와 동일한 epochs=20)
    print("\n=== 신경망 모델 학습 ===")
    print("모델 학습 중...")
    history = model.fit(x_train, demand_train, epochs=20, verbose=1, batch_size=1000)

    # 테스트 데이터 생성
    n_test = 100000
    temp_new = np.random.uniform(0, 30, n_test)
    weather_new = np.random.binomial(1, 0.2, n_test)  # 4/5 확률로 맑음, 1/5 확률로 흐림

    # 테스트 데이터 전처리
    x_test = np.zeros((n_test, 2), dtype='float32')
    x_test[:, 0] = (temp_new - min(temp)) / (max(temp) - min(temp))
    x_test[:, 1] = weather_new

    # 예측
    print("\n예측 중...")
    pred = model.predict(x_test, verbose=0)
    Q_prop = pred * (max(demand) - min(demand)) + min(demand)

    # 성능 평가
    sum_profit_nn = 0
    sum_Q_nn = 0

    print("성능 평가 중...")
    for i in range(n_test):
        # 실제 수요 계산
        if weather_new[i] == 0:  # 맑은 날
            mu_d = 100 + 3 * temp_new[i]
            sigma_d = 0.5 * mu_d
        else:  # 흐린 날
            mu_d = 30 + 4 * temp_new[i]
            sigma_d = 0.6 * mu_d

        d = np.random.normal(mu_d, sigma_d)
        if d < 0:
            d = 0

        Q = Q_prop[i][0]  # 예측된 생산량

        # 이익 계산
        profit = -c * Q
        if d <= Q:
            profit += p * d + s * (Q - d)
        else:
            profit += p * Q

        sum_Q_nn += Q
        sum_profit_nn += profit

    # 신경망 결과
    avg_Q_nn = sum_Q_nn / n_test
    avg_profit_nn = sum_profit_nn / n_test

    print(f"\n=== 신경망 모델 결과 ===")
    print(f"평균 생산량: {avg_Q_nn:.2f}")
    print(f"평균 기대이익: {avg_profit_nn:.2f}원")


# 비교를 위한 단순 평균 방법
sum_profit_simple = 0
sum_Q_simple = 0

print("\n=== 단순 평균 방법 (기준선) ===")
for i in range(10000):  # 비교를 위해 1만 개 샘플만 사용
    temp_sample = np.random.uniform(0, 30)
    weather_sample = np.random.binomial(1, 0.2)

    # 단순히 평균 수요로 생산량 결정
    if weather_sample == 0:  # 맑은 날
        Q = 100 + 3 * temp_sample
        mu_d = 100 + 3 * temp_sample
        sigma_d = 0.5 * mu_d
    else:  # 흐린 날
        Q = 30 + 4 * temp_sample
        mu_d = 30 + 4 * temp_sample
        sigma_d = 0.6 * mu_d

    d = np.random.normal(mu_d, sigma_d)
    if d < 0:
        d = 0

    # 이익 계산
    profit = -c * Q
    if d <= Q:
        profit += p * d + s * (Q - d)
    else:
        profit += p * Q

    sum_Q_simple += Q
    sum_profit_simple += profit

avg_Q_simple = sum_Q_simple / 10000
avg_profit_simple = sum_profit_simple / 10000

print(f"평균 생산량: {avg_Q_simple:.2f}")
print(f"평균 기대이익: {avg_profit_simple:.2f}원")

# 결과 비교
print(f"\n=== 결과 비교 ===")
print(f"단순 평균 방법: 생산량 {avg_Q_simple:.2f}, 이익 {avg_profit_simple:.2f}원")
if TF_AVAILABLE:
    print(f"신경망 방법:   생산량 {avg_Q_nn:.2f}, 이익 {avg_profit_nn:.2f}원")
    improvement = avg_profit_nn - avg_profit_simple
    improvement_pct = (improvement / avg_profit_simple) * 100
    print(f"이익 개선: {improvement:.2f}원 ({improvement_pct:.2f}%)")
else:
    print("TensorFlow를 설치하면 신경망 방법과 비교할 수 있습니다.")

# 조건별 이론적 수요 분석
print(f"\n=== 조건별 이론적 수요 분석 ===")
temp_samples = [0, 15, 30]  # 최저, 중간, 최고 온도

for t in temp_samples:
    print(f"\n온도 {t}°C에서:")

    # 맑은 날 수요
    mu_sunny = 100 + 3 * t
    sigma_sunny = 0.5 * mu_sunny
    print(f"  맑은 날 (80% 확률): 평균 {mu_sunny:.0f}, 표준편차 {sigma_sunny:.0f}")

    # 흐린 날 수요
    mu_cloudy = 30 + 4 * t
    sigma_cloudy = 0.6 * mu_cloudy
    print(f"  흐린 날 (20% 확률): 평균 {mu_cloudy:.0f}, 표준편차 {sigma_cloudy:.0f}")

    # 가중 평균 수요
    weighted_avg = 0.8 * mu_sunny + 0.2 * mu_cloudy
    print(f"  가중 평균 수요: {weighted_avg:.0f}")

print(f"총 시뮬레이션 데이터: {n:,}개")
if TF_AVAILABLE:
    print(f"신경망 테스트 데이터: {n_test:,}개")
    print(f"신경망 학습 에포크: 20")
print(f"단순 방법 테스트 데이터: 10,000개")